In [ ]:
# ==============================
# CBOW Implementation in PyTorch
# ==============================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import torch.nn.functional as F

# ------------------------------
# 1. Sample Dataset
# ------------------------------

sentences = [

    # Daily conversation
    "i am going to school",
    "i am going to office",
    "i am going to market",
    "i am very happy today",
    "i am feeling sad today",
    "how are you doing",
    "how is your day going",
    "what are you doing",
    "what is your name",
    "nice to meet you",

    # Common texting patterns
    "see you tomorrow",
    "see you soon",
    "talk to you later",
    "call me when you reach",
    "please call me back",
    "thank you so much",
    "thanks for your help",
    "sorry for the delay",
    "i will call you later",
    "i will text you later",

    # Shopping & travel
    "i want to buy a phone",
    "i want to buy new shoes",
    "i need some help",
    "can you help me",
    "where are you now",
    "i am on the way",
    "i reached home safely",
    "i will be late today",

    # Food
    "i love eating pizza",
    "i love eating burger",
    "let us order food",
    "what do you want to eat",
    "i am very hungry",
    "the food was delicious",

    # Study & work
    "i am studying machine learning",
    "i am preparing for exam",
    "i have completed my work",
    "please send me the file",
    "did you finish the task",
    "let us start the project",

    # Weather
    "the weather is very nice",
    "it is raining today",
    "it is very hot outside",
    "it is very cold today",

    # Emotions
    "i am excited for the trip",
    "i am nervous about the interview",
    "this is very important",
    "that sounds good",
    "that sounds interesting",

    # Extra variety
    "i forgot my wallet",
    "please remind me tomorrow",
    "let me know the details",
    "are you coming today",
    "i am waiting for you"
]

# ------------------------------
# 2. Preprocessing
# ------------------------------

# Tokenization
words = []
for sentence in sentences:
    words.extend(sentence.split())

vocab = list(set(words))
vocab_size = len(vocab)

# Word to index mapping
word_to_ix = {word: i for i, word in enumerate(vocab)}
ix_to_word = {i: word for word, i in word_to_ix.items()}

# ------------------------------
# 3. Create CBOW Training Data
# ------------------------------

context_size = 2
data = []

for sentence in sentences:
    tokens = sentence.split()
    for i in range(context_size, len(tokens) - context_size):
        context = []
        for j in range(-context_size, context_size + 1):
            if j != 0:
                context.append(tokens[i + j])
        target = tokens[i]
        data.append((context, target))

print("Sample Training Pair:")
print(data[0])

# ------------------------------
# 4. Define CBOW Model
# ------------------------------

class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOW, self).__init__()

        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(embedding_dim, 128)
        self.activation = nn.ReLU()
        self.linear2 = nn.Linear(128, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs)
        embeds = torch.mean(embeds, dim=0).view(1, -1)
        out = self.linear1(embeds)
        out = self.activation(out)
        out = self.linear2(out)
        return out

# ------------------------------
# 5. Training Setup
# ------------------------------

embedding_dim = 10
model = CBOW(vocab_size, embedding_dim)

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ------------------------------
# 6. Training Loop
# ------------------------------

for epoch in range(200):
    total_loss = 0

    for context, target in data:

        context_idxs = torch.tensor(
            [word_to_ix[w] for w in context],
            dtype=torch.long
        )

        model.zero_grad()

        output = model(context_idxs)

        loss = loss_function(
            output,
            torch.tensor([word_to_ix[target]], dtype=torch.long)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

# ------------------------------
# 7. Test Prediction Function
# ------------------------------

import torch.nn.functional as F

def predict_topk(context_words, k=3):

    # Check unknown words
    for word in context_words:
        if word not in word_to_ix:
            print(f"Word '{word}' not in vocabulary!")
            return

    context_idxs = torch.tensor(
        [word_to_ix[w] for w in context_words],
        dtype=torch.long
    )

    with torch.no_grad():
        output = model(context_idxs)
        probabilities = F.softmax(output, dim=1)

        topk_probs, topk_indices = torch.topk(probabilities, k)

        print(f"\nContext: {context_words}")
        print("Top Predictions:")

        for i in range(k):
            word = ix_to_word[topk_indices[0][i].item()]
            prob = topk_probs[0][i].item()
            print(f"{word}  -->  {prob:.4f}")

# ------------------------------
# 8. Try Prediction
# ------------------------------

print("\nPrediction Example:")
print("Context: ['i', 'love', 'learning', 'is']")
print("Predicted Word:", predict(['i', 'love', 'learning', 'is']))

Sample Training Pair:
(['i', 'am', 'to', 'school'], 'going')
Epoch 0, Loss: 156.0635
Epoch 20, Loss: 1.5939
Epoch 40, Loss: 1.5104
Epoch 60, Loss: 1.4837
Epoch 80, Loss: 1.4670
Epoch 100, Loss: 1.4556
Epoch 120, Loss: 1.4447
Epoch 140, Loss: 1.4353
Epoch 160, Loss: 1.4274
Epoch 180, Loss: 1.4205

Prediction Example:
Context: ['i', 'love', 'learning', 'is']
Predicted Word: your


In [ ]:
while True:
    text = input("\nEnter 4 context words separated by space (or type 'exit'): ")

    if text.lower() == 'exit':
        break

    context_words = text.split()

    if len(context_words) != 4:
        print("Please enter exactly 4 words!")
        continue

    predict_topk(context_words, k=3)


Enter 4 context words separated by space (or type 'exit'): i am to school

Context: ['i', 'am', 'to', 'school']
Top Predictions:
going  -->  1.0000
nervous  -->  0.0000
want  -->  0.0000

Enter 4 context words separated by space (or type 'exit'): i am for the

Context: ['i', 'am', 'for', 'the']
Top Predictions:
excited  -->  1.0000
preparing  -->  0.0000
waiting  -->  0.0000

Enter 4 context words separated by space (or type 'exit'): the weather is very

Context: ['the', 'weather', 'is', 'very']
Top Predictions:
is  -->  0.4646
your  -->  0.4535
very  -->  0.0520

Enter 4 context words separated by space (or type 'exit'): i love eating
Please enter exactly 4 words!

Enter 4 context words separated by space (or type 'exit'): how is your day

Context: ['how', 'is', 'your', 'day']
Top Predictions:
your  -->  1.0000
me  -->  0.0000
very  -->  0.0000

Enter 4 context words separated by space (or type 'exit'): how is day going

Context: ['how', 'is', 'day', 'going']
Top Predictions:
your  -

KeyboardInterrupt: Interrupted by user

In [ ]:
!pip install fastapi uvicorn pyngrok nest-asyncio nltk pydantic

In [3]:
# ==============================
# SMART E-COMMERCE SEARCH (COLAB VERSION - NO NGROK)
# ==============================

from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
from nltk.wsd import lesk
from nltk.tokenize import word_tokenize
import nltk
import uvicorn
import nest_asyncio
import threading
import requests
import time

# Allow FastAPI inside notebook
nest_asyncio.apply()

# Download NLTK Data
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

# ==============================
# Models
# ==============================

class Product(BaseModel):
    id: int
    name: str
    category: str
    description: str

class SearchResponse(BaseModel):
    query: str
    detected_category: str | None
    total_results: int
    results: List[Product]

# ==============================
# Fake Database
# ==============================

products_db = [
    Product(id=1, name="Apple iPhone 15", category="electronics",
            description="Latest Apple smartphone device"),

    Product(id=2, name="Apple MacBook Pro", category="electronics",
            description="Apple laptop with M3 chip"),

    Product(id=3, name="iPhone Charger", category="electronics",
            description="Fast charging adapter"),

    Product(id=4, name="Fresh Red Apple", category="grocery",
            description="Sweet and fresh fruit apple"),

    Product(id=5, name="Organic Green Apple", category="grocery",
            description="Healthy green fruit apple"),

    Product(id=6, name="Bananas", category="grocery",
            description="Fresh yellow bananas"),
]

# ==============================
# WSD Logic
# ==============================

def detect_apple_sense(query: str):
    try:
        words = word_tokenize(query.lower())
        sense = lesk(words, "apple")

        if not sense:
            return None

        definition = sense.definition().lower()

        if "fruit" in definition:
            return "grocery"

        if "company" in definition or "computer" in definition:
            return "electronics"

        return None

    except Exception as e:
        print("WSD Error:", e)
        return None

# ==============================
# Intelligent Search
# ==============================

def intelligent_search(query: str):
    query_lower = query.lower()
    detected_category = None

    if "apple" in query_lower:
        detected_category = detect_apple_sense(query_lower)

    ranked_results = []

    for product in products_db:

        if detected_category and product.category != detected_category:
            continue

        score = 0

        for word in query_lower.split():
            if word in product.name.lower():
                score += 2
            if word in product.description.lower():
                score += 1

        if score > 0:
            ranked_results.append((score, product))

    ranked_results.sort(key=lambda x: x[0], reverse=True)

    return detected_category, [product for _, product in ranked_results]

# ==============================
# FastAPI App
# ==============================

app = FastAPI(title="Smart Search API")

@app.get("/search", response_model=SearchResponse)
def search(query: str):
    try:
        detected_category, results = intelligent_search(query)

        return SearchResponse(
            query=query,
            detected_category=detected_category,
            total_results=len(results),
            results=results
        )

    except Exception as e:
        print("Search Error:", e)
        return SearchResponse(
            query=query,
            detected_category=None,
            total_results=0,
            results=[]
        )

# ==============================
# Run Server in Background
# ==============================

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run)
thread.daemon = True
thread.start()

time.sleep(2)

print("✅ FastAPI server started inside Colab!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
INFO:     Started server process [274]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


✅ FastAPI server started inside Colab!


In [5]:
# Test 1
import nltk
nltk.download('punkt_tab')
response = requests.get("http://127.0.0.1:8000/search?query=apple phone")
print(response.json())

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


INFO:     127.0.0.1:37414 - "GET /search?query=apple%20phone HTTP/1.1" 200 OK
{'query': 'apple phone', 'detected_category': 'grocery', 'total_results': 2, 'results': [{'id': 4, 'name': 'Fresh Red Apple', 'category': 'grocery', 'description': 'Sweet and fresh fruit apple'}, {'id': 5, 'name': 'Organic Green Apple', 'category': 'grocery', 'description': 'Healthy green fruit apple'}]}


In [6]:
response = requests.get("http://127.0.0.1:8000/search?query=apple phone")
print(response.status_code)
print(response.json())

INFO:     127.0.0.1:54310 - "GET /search?query=apple%20phone HTTP/1.1" 200 OK
200
{'query': 'apple phone', 'detected_category': 'grocery', 'total_results': 2, 'results': [{'id': 4, 'name': 'Fresh Red Apple', 'category': 'grocery', 'description': 'Sweet and fresh fruit apple'}, {'id': 5, 'name': 'Organic Green Apple', 'category': 'grocery', 'description': 'Healthy green fruit apple'}]}


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 29.0 MB/s eta 0:00:00


In [ ]:
from gensim.models import Word2Vec

print("🚔 Crime Prediction System Started\n")


data = [
    ["late_night", "isolated_area", "suspicious_activity", "theft"],
    ["crowded_area", "pickpocket", "escape"],
    ["dark_street", "no_cameras", "robbery"],
    ["late_night", "alcohol", "fight", "violence"],
    ["isolated_area", "vehicle_break", "theft"],
    ["market", "crowd", "pickpocket"],
    ["dark_street", "suspicious_activity", "robbery"],
    ["late_night", "isolated_area", "robbery"],
]

print("✅ Data loaded")


model = Word2Vec(
    sentences=data,
    vector_size=50,
    window=2,
    min_count=1,
    sg=1   # Skip-Gram
)

print("✅ Model trained\n")



print("🔍 Related to 'late_night':")
print(model.wv.most_similar("late_night"))

print("\n🔍 Related to 'isolated_area':")
print(model.wv.most_similar("isolated_area"))

print("\n🔍 Related to 'suspicious_activity':")
print(model.wv.most_similar("suspicious_activity"))


print("\n🧠 Try your own input")
user_input = input("Enter situation (e.g., late_night): ")

if user_input in model.wv:
    print("\n🚨 Possible related risks:")
    print(model.wv.most_similar(user_input))
else:
    print("❌ Not found in dataset")


model.save("crime_skipgram.model")

print("\n💾 Model saved")
print("🚀 Done!")

🚔 Crime Prediction System Started

✅ Data loaded
✅ Model trained

🔍 Related to 'late_night':
[('pickpocket', 0.16563551127910614), ('fight', 0.1551763415336609), ('crowded_area', 0.14387670159339905), ('market', 0.13940520584583282), ('robbery', 0.12669356167316437), ('violence', 0.12119622528553009), ('suspicious_activity', 0.08872983604669571), ('escape', 0.032278481870889664), ('no_cameras', 0.02048538811504841), ('isolated_area', 0.011071977205574512)]

🔍 Related to 'isolated_area':
[('pickpocket', 0.12486250698566437), ('alcohol', 0.08061248809099197), ('suspicious_activity', 0.07399576157331467), ('robbery', 0.04237452521920204), ('theft', 0.018277151510119438), ('escape', 0.011398451402783394), ('late_night', 0.011071980930864811), ('vehicle_break', 0.0013571369927376509), ('no_cameras', -0.01201754529029131), ('crowded_area', -0.03442405164241791)]

🔍 Related to 'suspicious_activity':
[('theft', 0.18339456617832184), ('violence', 0.16944658756256104), ('crowd', 0.11253629624843

KeyboardInterrupt: Interrupted by user